# 실습 세트 1. 상관분석을 활용한 반도체 생산품질 중요 요인 찾기

## 1. CSV 파일 다운로드

먼저 아래 CSV 파일을 다운로드한 뒤, 노트북 파일과 같은 폴더에 둡니다.

파일명:

`semicond_preprocess.csv`

이 데이터는 반도체 공정 센서값과 제품의 정상/불량 여부를 담고 있는 가상 데이터입니다.

---

## 2. 문제 상황 설명

반도체 생산 공정에서는 온도, 압력, 가스 유량, 파티클 수, 두께 균일도, 전압 안정성 등 다양한 센서 데이터가 수집됩니다.

이번 실습에서는 각 생산 기록별 센서값과 제품의 불량 여부가 주어져 있습니다.

`Defect` 컬럼은 제품의 정상/불량 여부를 의미합니다.

- `Defect = 0`: 정상
- `Defect = 1`: 불량

이번 분석의 목표는 pandas와 seaborn을 사용하여 어떤 공정 센서값이 불량 여부와 관련이 큰지 탐색하는 것입니다.

단, 상관관계가 높다고 해서 반드시 그 변수가 불량의 원인이라는 뜻은 아닙니다.  
상관분석은 불량과 함께 움직이는 후보 변수를 찾는 탐색 도구로 이해해야 합니다.

### 데이터 컬럼 설명

| 컬럼명 | 설명 |
|---|---|
| Wafer_ID | 웨이퍼 ID |
| Lot_ID | 생산 Lot ID |
| Temperature | 공정 온도 |
| Chamber_Pressure | 챔버 압력 |
| Gas_Flow | 가스 유량 |
| Etch_Rate | 식각 속도 |
| Chamber_Humidity | 챔버 습도 |
| Particle_Count | 공정 중 파티클 수 |
| Thickness_Uniformity | 박막 두께 균일도 |
| Voltage_Stability | 전압 안정성 |
| Defect | 정상/불량 여부 |



# 문제 1. 데이터 기본 확인

CSV 파일을 불러온 뒤 다음 내용을 확인하세요.

- 상위 5개 행
- 데이터 크기
- 컬럼별 자료형
- 결측치 개수
- 기초 통계량

수치형 컬럼뿐 아니라 문자형 컬럼까지 함께 확인할 수 있도록 `describe(include="all")`도 사용해보세요.


In [3]:
import pandas as pd

df = pd.read_csv("semicond_preprocess.csv")
df.isna().sum()

Wafer_ID                 0
Lot_ID                   0
Factory                  0
Shift                    0
Process_Mode            27
Product_Type             0
Material_Supplier       45
Chamber_ID               0
Temperature              0
Chamber_Pressure        63
Gas_Flow                81
Etch_Rate                0
Chamber_Humidity        54
Particle_Count           0
Thickness_Uniformity    72
Voltage_Stability       90
Cycle_Time               0
Defect                   0
dtype: int64

# 문제 2. 조건에 맞는 데이터 필터링

다음 조건에 해당하는 데이터를 각각 필터링하세요.

1. 불량 제품만 선택하세요.
2. 야간 교대조(`Night`)에서 생산된 제품만 선택하세요.
3. `Factory`가 C이고 `Product_Type`이 Premium인 제품만 선택하세요.

각 필터링 결과의 데이터 개수도 함께 출력하세요.

In [5]:
df[df['Defect'] == 1].head()

,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
14,WAFER-00015,LOT-033,A,Day,Normal,Premium,S2,CH-04,485.90,NaN,113.77,1.575,48.78,12.6,0.8952,0.9655,34.27,1
20,WAFER-00021,LOT-017,A,Night,High_Speed,Standard,S1,CH-04,475.81,2.458,128.66,1.613,31.62,85.0,0.9477,0.9681,34.29,1
39,WAFER-00040,LOT-005,A,Day,High_Speed,Advanced,S2,CH-01,470.32,1.392,116.43,1.677,38.47,20.4,0.9163,0.9956,24.95,1
42,WAFER-00043,LOT-007,C,Night,Normal,Advanced,S1,CH-04,478.13,2.785,131.35,1.598,NaN,29.3,0.9396,0.9578,31.01,1
52,WAFER-00053,LOT-029,B,Day,High_Speed,Advanced,S1,CH-03,495.92,2.700,118.32,1.749,30.15,16.6,0.9053,0.9704,23.93,1


In [6]:
df[df['Shift'] == 'Night'].head()

,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
1,WAFER-00002,LOT-035,B,Night,Normal,Standard,S1,CH-02,458.02,2.543,117.65,1.574,32.49,14.1,0.9506,0.9740,28.96,0
5,WAFER-00006,LOT-002,C,Night,Normal,Standard,S1,CH-03,480.79,2.214,115.73,1.599,48.44,25.4,0.9489,0.9757,30.10,0
10,WAFER-00011,LOT-025,B,Night,Normal,Advanced,S1,CH-02,512.93,2.277,134.35,1.522,35.44,22.2,0.8998,0.9528,27.45,0
11,WAFER-00012,LOT-004,A,Night,High_Speed,Advanced,S1,CH-04,467.51,2.224,118.79,1.516,40.81,19.9,0.9427,0.9691,24.13,0
18,WAFER-00019,LOT-019,C,Night,Normal,Premium,S3,CH-01,469.13,2.495,109.61,1.467,36.61,17.7,0.9191,0.9715,27.65,0


In [7]:
df[ (df['Factory'] == 'C') & (df['Product_Type'] == 'Premium') ].head()

,Wafer_ID,Lot_ID,Factory,Shift,Process_Mode,Product_Type,Material_Supplier,Chamber_ID,Temperature,Chamber_Pressure,Gas_Flow,Etch_Rate,Chamber_Humidity,Particle_Count,Thickness_Uniformity,Voltage_Stability,Cycle_Time,Defect
18,WAFER-00019,LOT-019,C,Night,Normal,Premium,S3,CH-01,469.13,2.495,109.61,1.467,36.61,17.7,0.9191,0.9715,27.65,0
29,WAFER-00030,LOT-033,C,Day,Normal,Premium,S1,CH-04,481.52,2.372,122.54,1.576,38.50,22.6,0.9132,0.9760,33.20,0
58,WAFER-00059,LOT-006,C,Day,Normal,Premium,S2,CH-01,482.37,2.231,117.65,1.509,30.48,14.9,0.9133,0.9750,32.54,0
76,WAFER-00077,LOT-024,C,Day,Normal,Premium,S1,CH-03,493.46,2.408,117.16,1.646,45.64,25.2,0.9518,0.9648,32.02,0
81,WAFER-00082,LOT-009,C,Day,High_Speed,Premium,S3,CH-03,462.46,2.730,125.11,1.439,NaN,25.9,0.9115,0.9883,25.58,1


# 문제 3. 상관계수를 구하여 변수 선택하기

수치형 컬럼만 선택한 뒤 상관계수를 계산하세요.

그다음 `Defect`와 다른 수치형 변수들의 상관계수를 구하고, 절댓값 기준으로 정렬하세요.

`Defect`와 관련이 높은 상위 5개 변수를 선택하세요.


In [24]:
df_corr = pd.DataFrame(df.select_dtypes(include='number').corr()['Defect'])
df_corr['abs'] = df_corr['Defect'].abs()
df_corr.sort_values('abs', ascending=False).drop('Defect').iloc[:5].index

Index(['Particle_Count', 'Thickness_Uniformity', 'Voltage_Stability',
       'Gas_Flow', 'Cycle_Time'],
      dtype='str')

- **<font color="red">AI 코드</font>**

In [29]:
# -----------------------------
# 1. 수치형 컬럼만 선택
# -----------------------------
numeric_df = df.select_dtypes(include=['number'])

# -----------------------------
# 2. 상관계수 계산
# -----------------------------
corr_matrix = numeric_df.corr()

print("\n[상관계수 행렬]")
print(corr_matrix)

# -----------------------------
# 3. Defect 와의 상관계수 추출
# -----------------------------
defect_corr = corr_matrix["Defect"].drop("Defect")

# 절댓값 기준 정렬
defect_corr_sorted = defect_corr.reindex(
    defect_corr.abs().sort_values(ascending=False).index
)

print("\n[Defect 와의 상관계수]")
print(defect_corr_sorted)

# -----------------------------
# 4. 상위 5개 변수 선택
# -----------------------------
top5_features = defect_corr_sorted.head(5)

print("\n[Defect 관련 상위 5개 변수]")
print(top5_features)


[상관계수 행렬]
                      Temperature  Chamber_Pressure  Gas_Flow  Etch_Rate  \
Temperature              1.000000         -0.000388 -0.033630  -0.007711   
Chamber_Pressure        -0.000388          1.000000  0.045702   0.004955   
Gas_Flow                -0.033630          0.045702  1.000000  -0.007065   
Etch_Rate               -0.007711          0.004955 -0.007065   1.000000   
Chamber_Humidity        -0.004874          0.009530 -0.014535  -0.049491   
Particle_Count          -0.041312          0.030202 -0.030941  -0.030269   
Thickness_Uniformity     0.012298          0.021970 -0.000148   0.039620   
Voltage_Stability       -0.041218         -0.023087 -0.017461   0.019068   
Cycle_Time              -0.100458          0.030952  0.029578   0.030183   
Defect                  -0.038028         -0.018410 -0.071142  -0.015401   

                      Chamber_Humidity  Particle_Count  Thickness_Uniformity  \
Temperature                  -0.004874       -0.041312              0.01

# 문제 4. 결측치 처리

결측치가 있는 컬럼을 확인한 뒤, 다음 방식으로 결측치를 처리하세요.

- 수치형 컬럼의 결측치는 중앙값으로 채우세요.
- 문자형 컬럼의 결측치는 `"Unknown"`으로 채우세요.
- 결측치 처리 후 결측치가 남아 있는지 다시 확인하세요.

원본 데이터는 보존하고, 결측치 처리 결과는 `df_clean`이라는 새 데이터프레임에 저장하세요.

In [19]:
df_clean = df.copy()

# df_clean.isnull().sum()
num_cols = df_clean.select_dtypes(include='number').columns
cat_cols = df_clean.select_dtypes(exclude='number').columns

num_fill_values = df_clean[num_cols].median()
cat_fill_values = "Unknown"

df_clean[num_cols] = df_clean[num_cols].fillna(num_fill_values)
df_clean[cat_cols] = df_clean[cat_cols].fillna(cat_fill_values)

df_clean.isnull().sum()

Wafer_ID                0
Lot_ID                  0
Factory                 0
Shift                   0
Process_Mode            0
Product_Type            0
Material_Supplier       0
Chamber_ID              0
Temperature             0
Chamber_Pressure        0
Gas_Flow                0
Etch_Rate               0
Chamber_Humidity        0
Particle_Count          0
Thickness_Uniformity    0
Voltage_Stability       0
Cycle_Time              0
Defect                  0
dtype: int64

- **<font color="red">AI 코드</font>**

In [30]:
# -----------------------------------
# 1. 원본 보존 후 복사본 생성
# -----------------------------------
df_clean = df.copy()

# -----------------------------------
# 2. 결측치 확인
# -----------------------------------
print("[결측치 개수]")
print(df_clean.isnull().sum())

# -----------------------------------
# 3. 수치형 / 문자형 컬럼 분리
# -----------------------------------
numeric_cols = df_clean.select_dtypes(include=['number']).columns
object_cols = df_clean.select_dtypes(include=['object']).columns

# -----------------------------------
# 4. 수치형 결측치 → 중앙값 대체
# -----------------------------------
for col in numeric_cols:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)

# -----------------------------------
# 5. 문자형 결측치 → "Unknown" 대체
# -----------------------------------
for col in object_cols:
    df_clean[col] = df_clean[col].fillna("Unknown")

# -----------------------------------
# 6. 결측치 처리 후 재확인
# -----------------------------------
print("\n[결측치 처리 후]")
print(df_clean.isnull().sum())

# 전체 결측치 개수 확인
print("\n전체 남은 결측치 수:")
print(df_clean.isnull().sum().sum())

[결측치 개수]
Wafer_ID                 0
Lot_ID                   0
Factory                  0
Shift                    0
Process_Mode            27
Product_Type             0
Material_Supplier       45
Chamber_ID               0
Temperature              0
Chamber_Pressure        63
Gas_Flow                81
Etch_Rate                0
Chamber_Humidity        54
Particle_Count           0
Thickness_Uniformity    72
Voltage_Stability       90
Cycle_Time               0
Defect                   0
dtype: int64

[결측치 처리 후]
Wafer_ID                0
Lot_ID                  0
Factory                 0
Shift                   0
Process_Mode            0
Product_Type            0
Material_Supplier       0
Chamber_ID              0
Temperature             0
Chamber_Pressure        0
Gas_Flow                0
Etch_Rate               0
Chamber_Humidity        0
Particle_Count          0
Thickness_Uniformity    0
Voltage_Stability       0
Cycle_Time              0
Defect                  0
dtype: int64

C:\Users\metam\AppData\Local\Temp\ipykernel_9888\1872106929.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df_clean.select_dtypes(include=['object']).columns


# 문제 5. IQR 방식으로 이상치 처리하기

`Particle_Count` 컬럼을 대상으로 IQR 방식의 이상치 기준을 계산하세요.

다음 값을 구하세요.

- Q1
- Q3
- IQR
- lower bound
- upper bound

그다음 `Particle_Count`가 IQR 기준을 벗어난 행을 찾으세요.


1. 이상치는 몇개인가요?
2. 이상치를 가지는 행을 삭제하세요.

In [28]:
col_name = "Particle_Count"

Q1 = df[col_name].quantile(0.25)
Q3 = df[col_name].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5*IQR
lower_bound = Q1 - 1.5*IQR

outlier = (df[col_name] < lower_bound) | (df[col_name] > upper_bound)
print(outlier.sum())
df[~outlier].shape


39


(1761, 18)

- **<font color="red">AI 코드</font>**

In [27]:
# -----------------------------------
# 1. Q1, Q3 계산
# -----------------------------------
Q1 = df["Particle_Count"].quantile(0.25)
Q3 = df["Particle_Count"].quantile(0.75)

# -----------------------------------
# 2. IQR 계산
# -----------------------------------
IQR = Q3 - Q1

# -----------------------------------
# 3. 이상치 경계 계산
# -----------------------------------
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

# 결과 출력
print("Q1 :", Q1)
print("Q3 :", Q3)
print("IQR :", IQR)
print("Lower Bound :", lower_bound)
print("Upper Bound :", upper_bound)

# -----------------------------------
# 4. 이상치 찾기
# -----------------------------------
outliers = df[
    (df["Particle_Count"] < lower_bound) |
    (df["Particle_Count"] > upper_bound)
]

print("\n[이상치 행]")
print(outliers)

# 이상치 개수
print("\n이상치 개수 :", len(outliers))

# -----------------------------------
# 5. 이상치 제거
# -----------------------------------
df_no_outlier = df[
    (df["Particle_Count"] >= lower_bound) &
    (df["Particle_Count"] <= upper_bound)
]

print("\n이상치 제거 후 데이터 크기 :", df_no_outlier.shape)

Q1 : 16.3
Q3 : 22.9
IQR : 6.599999999999998
Lower Bound : 6.400000000000004
Upper Bound : 32.8

[이상치 행]
         Wafer_ID   Lot_ID Factory  Shift Process_Mode Product_Type  \
20    WAFER-00021  LOT-017       A  Night   High_Speed     Standard   
92    WAFER-00093  LOT-023       C    Day       Normal     Standard   
173   WAFER-00174  LOT-014       C    Day       Normal     Advanced   
187   WAFER-00188  LOT-026       A    Day   High_Speed     Standard   
214   WAFER-00215  LOT-022       B    Day       Normal     Standard   
245   WAFER-00246  LOT-030       A    Day   High_Speed     Advanced   
289   WAFER-00290  LOT-030       A    Day   High_Speed     Advanced   
390   WAFER-00391  LOT-018       A    Day       Normal     Standard   
438   WAFER-00439  LOT-018       B  Night   High_Speed     Standard   
565   WAFER-00566  LOT-032       B    Day       Normal     Standard   
586   WAFER-00587  LOT-029       C  Night       Normal     Standard   
661   WAFER-00662  LOT-015       C  Night   